In [49]:
import pandas as pd
import numpy as np


In [50]:
train_path = "../data/processed_data/train_preprocessed.csv"
test_path = "../data/processed_data/test_preprocessed.csv"

In [51]:
data_train = pd.read_csv(train_path)
data_test = pd.read_csv(test_path)

In [52]:
col_char = ['State', 'International plan', 'Voice mail plan']

In [41]:
## eliminating the correlated features

import numpy as np

corr_matrix = data_train.drop(col_char, axis = 1).corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper.columns if any(upper[column] > 0.8)
]

print(high_corr_features)

['Total day charge', 'Total eve charge', 'Total night charge', 'Total intl charge']


In [42]:
data_train['Churn']

0       0
1       0
2       0
3       0
4       0
       ..
2661    0
2662    0
2663    0
2664    0
2665    0
Name: Churn, Length: 2666, dtype: int64

In [54]:
import pandas as pd
from sklearn.preprocessing import StandardScaler


def feature_engineering(df, test_flag = 0):

    df = df.copy()

    # 1 Convert Boolean / Yes-No to Numeric

    if df["International plan"].dtype == "object":
        df["International plan"] = df["International plan"].map({"Yes":1,"No":0})

    if df["Voice mail plan"].dtype == "object":
        df["Voice mail plan"] = df["Voice mail plan"].map({"Yes":1,"No":0})


    # 2 Drop High Cardinality Column

    if "State" in df.columns:
        df = df.drop("State", axis=1)


    # 3 Remove Highly Correlated Features
    # Charges are derived from minutes

    correlated_features = [
        "Total day charge",
        "Total eve charge",
        "Total night charge",
        "Total intl charge"
    ]

    for col in correlated_features:
        if col in df.columns:
            df = df.drop(col, axis=1)


    # 4 Create New Useful Features

    df["Total minutes"] = (
        df["Total day minutes"]
        + df["Total eve minutes"]
        + df["Total night minutes"]
        + df["Total intl minutes"]
    )

    df["Total calls"] = (
        df["Total day calls"]
        + df["Total eve calls"]
        + df["Total night calls"]
        + df["Total intl calls"]
    )


    # 5 Feature Scaling

    # scaler = StandardScaler()

    # numeric_cols = df.select_dtypes(include=["int64","float64"]).columns
    # # numeric_cols = numeric_cols.drop("Churn")
    # df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    

    return df

In [55]:
data_train = feature_engineering(data_train)
data_test = feature_engineering(data_test)

In [56]:
data_train.to_csv("../data/processed_data/train_preprocessed.csv")
data_test.to_csv("../data/processed_data/test_preprocessed.csv")


In [57]:
data_train.Churn.unique()

array([0, 1], dtype=int64)

In [ ]:
def correlated_features(df, column_char, threshold):
    corr_matrix = df.drop(column_char, axis = 1).corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k = 1).astype(bool))
    high_corr_features = [column for column in upper.columns if any(upper[column] > 0.8)]
    return high_corr_features
